# Wetland drainage areas: first wetland reached along the D8 path

**Pass 1.** Each cell follows its D8 steepest-descent path until it reaches a wetland, leaves the DEM, or ends in a closed depression (pit). Flats drain to their nearest exit at the same elevation (a lower cell, a wetland, or the DEM edge). Each pit gets a unique ID, and every cell draining into it is labelled with that ID.

**Pass 2.** Pits are drained from the lowest outlet upward. A pit's outlet is the lowest point where water can cross out of its catchment (the higher of the two cells on either side of the boundary). Equally low outlets: the steepest drop wins; exact ties go to the inside cell that comes first in raster order, then the first neighbour direction (NE, E, SE, S, SW, W, NW, N). Every cell in the pit takes the label of the cell across the outlet. If that cell belongs to a pit that hasn't drained yet, the two pits merge and drain together through their next outlet.

| value | pass 1 (`_pass1.tif`) | final (`_labels.tif`) |
|---|---|---|
| ≥ 1 | wetland ID + 1 | wetland ID + 1 |
| 0 | reaches a stream (if `STREAMS_PATH` is set) or leaves the DEM, without reaching a wetland | same |
| ≤ -1 | minus the pit ID | (none left) |
| -2147483648 | nodata | nodata |

Memory: about 10 bytes per DEM cell plus ~1 GB. Requirements: `pip install numba rasterio geopandas pyogrio`

In [ ]:
DEM_PATH       = r"/path/to/17110009_Skykomish_DEM_3m_huc8.tif"
WETLANDS_PATH  = r"/path/to/tw_wetland_polygons_mid.gpkg"
WETLANDS_LAYER = None            # layer name inside the GeoPackage, or None
ID_FIELD       = "Wetlands ID"   # non-negative integers
OUT_DIR        = r"/path/to/wetland_drainage_out"
PREFIX         = "skykomish"
ALL_TOUCHED    = False           # True burns every cell a polygon touches (keeps sub-pixel wetlands)
WRITE_POLYGONS = False           # dissolved drainage-area polygons from the final raster (slow on big rasters)
STREAMS_PATH   = None            # flowlines shapefile; their cells become 0, so flow reaching them bypasses wetlands

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from numba import njit, types
from numba.typed import Dict
from rasterio import features
from shapely.geometry import shape

EXTERNAL = 0                     # leaves the DEM without reaching a wetland
NODATA = -2147483648
UNSET = -2147483647              # not labelled yet (internal)
DR = np.array([-1, 0, 1, 1, 1, 0, -1, -1])   # neighbours NE, E, SE, S, SW, W, NW, N
DC = np.array([1, 1, 1, 0, -1, -1, -1, 0])
FLAT, OUT, WET = 8, 9, 10


@njit
def pass1(z, lab, dx, dy):
    """D8 routing. Fills lab in place; returns pit floor locations and sizes."""
    nr, nc = z.shape
    dist = np.sqrt((DR * dy) ** 2 + (DC * dx) ** 2)
    dirs = np.full((nr, nc), 255, dtype=np.uint8)
    d, l = dirs.ravel(), lab.ravel()             # flat views

    # steepest strictly-lower neighbour; edge cells with none drain out of the DEM
    n_flat = 0
    for r in range(nr):
        for c in range(nc):
            if np.isnan(z[r, c]):
                continue
            if lab[r, c] != UNSET:          # wetland or stream: flow stops here
                dirs[r, c] = WET
                continue
            edge = r == 0 or r == nr - 1 or c == 0 or c == nc - 1
            best, best_s = -1, 0.0
            for k in range(8):
                rr, cc = r + DR[k], c + DC[k]
                if rr < 0 or rr >= nr or cc < 0 or cc >= nc:
                    continue
                if np.isnan(z[rr, cc]):
                    edge = True
                    continue
                s = (z[r, c] - z[rr, cc]) / dist[k]
                if s > best_s:
                    best, best_s = k, s
            if best >= 0:
                dirs[r, c] = best
            elif edge:
                dirs[r, c] = OUT
                lab[r, c] = EXTERNAL
            else:
                dirs[r, c] = FLAT
                n_flat += 1

    # flats: breadth-first from same-elevation exits (stored as k + 16 until the scan ends,
    # so a flat cell assigned during the scan isn't mistaken for an exit)
    queue = np.empty(max(n_flat, 1), dtype=np.int64)
    head = tail = 0
    for r in range(nr):
        for c in range(nc):
            if dirs[r, c] != FLAT:
                continue
            for k in range(8):
                rr, cc = r + DR[k], c + DC[k]
                if 0 <= rr < nr and 0 <= cc < nc and z[rr, cc] == z[r, c] and \
                        (dirs[rr, cc] <= 7 or dirs[rr, cc] == OUT or dirs[rr, cc] == WET):
                    dirs[r, c] = k + 16
                    queue[tail] = r * nc + c
                    tail += 1
                    break
    for t in range(tail):
        d[queue[t]] -= 16
    while head < tail:
        r, c = divmod(queue[head], nc)
        head += 1
        for k in range(8):
            rr, cc = r + DR[k], c + DC[k]
            if 0 <= rr < nr and 0 <= cc < nc and dirs[rr, cc] == FLAT and z[rr, cc] == z[r, c]:
                dirs[rr, cc] = (k + 4) % 8
                queue[tail] = rr * nc + cc
                tail += 1

    # pits: each remaining closed flat or single cell gets an ID; its cells are labelled -ID
    n_pits = 0
    pit_cell = np.empty(n_flat + 1, dtype=np.int64)
    pit_floor = np.zeros(n_flat + 1, dtype=np.int64)
    for i in range(nr * nc):
        if d[i] != FLAT:
            continue
        n_pits += 1
        pit_cell[n_pits] = i
        d[i] = WET                  # closed from here on
        l[i] = -n_pits
        head, tail = 0, 1
        queue[0] = i
        while head < tail:
            r, c = divmod(queue[head], nc)
            head += 1
            pit_floor[n_pits] += 1
            for k in range(8):
                rr, cc = r + DR[k], c + DC[k]
                if 0 <= rr < nr and 0 <= cc < nc and dirs[rr, cc] == FLAT and z[rr, cc] == z[r, c]:
                    dirs[rr, cc] = WET
                    lab[rr, cc] = -n_pits
                    queue[tail] = rr * nc + cc
                    tail += 1

    # every other cell: follow the path down to the first labelled cell, then label the whole path
    off = DR * nc + DC
    path = np.empty(1 << 16, dtype=np.int64)
    for i in range(nr * nc):
        if l[i] != UNSET or d[i] > 7:
            continue
        n, j = 0, i
        while l[j] == UNSET:
            if n == path.size:
                path = np.concatenate((path, np.empty_like(path)))
            path[n] = j
            n += 1
            j += off[d[j]]
        for t in range(n):
            l[path[t]] = l[j]
    return n_pits, pit_cell[1:n_pits + 1], pit_floor[1:n_pits + 1]


@njit
def find(parent, a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]
        a = parent[a]
    return a


@njit
def pass2(z, lab, n_pits, dx, dy):
    """Drain pits from the lowest outlet up. Relabels pit cells in lab; returns per-pit outlet info."""
    nr, nc = z.shape

    # lowest crossing from each pit catchment into each neighbouring label
    slot = Dict.empty(types.int64, types.int64)
    cap = 1 << 16
    e_pit = np.empty(cap, np.int64)
    e_to = np.empty(cap, np.int64)
    e_z = np.empty(cap, np.float64)
    e_slope = np.empty(cap, np.float64)
    e_cell = np.empty(cap, np.int64)
    e_k = np.empty(cap, np.int64)
    dist = np.sqrt((DR * dy) ** 2 + (DC * dx) ** 2)
    n = 0
    for r in range(nr):
        for c in range(nc):
            p = -np.int64(lab[r, c])
            if p <= 0 or lab[r, c] == NODATA:
                continue
            for k in range(8):
                rr, cc = r + DR[k], c + DC[k]
                if rr < 0 or rr >= nr or cc < 0 or cc >= nc or lab[rr, cc] == NODATA:
                    to, s, g = EXTERNAL, z[r, c], 0.0        # spills off the DEM here
                else:
                    to = np.int64(lab[rr, cc])
                    if to == -p:
                        continue
                    s = max(z[r, c], z[rr, cc])
                    g = (z[r, c] - z[rr, cc]) / dist[k]     # ties: steepest way out wins
                key = (p << 32) | (to & 0xFFFFFFFF)
                if key not in slot:
                    if n == cap:
                        cap *= 2
                        e_pit = np.concatenate((e_pit, np.empty(n, np.int64)))
                        e_to = np.concatenate((e_to, np.empty(n, np.int64)))
                        e_z = np.concatenate((e_z, np.empty(n, np.float64)))
                        e_slope = np.concatenate((e_slope, np.empty(n, np.float64)))
                        e_cell = np.concatenate((e_cell, np.empty(n, np.int64)))
                        e_k = np.concatenate((e_k, np.empty(n, np.int64)))
                    slot[key] = n
                    e_pit[n], e_to[n], e_z[n], e_slope[n], e_cell[n], e_k[n] = p, to, s, g, r * nc + c, k
                    n += 1
                else:
                    j = slot[key]
                    if s < e_z[j] or (s == e_z[j] and g > e_slope[j]):
                        e_z[j], e_slope[j], e_cell[j], e_k[j] = s, g, r * nc + c, k

    # take crossings from lowest to highest
    parent = np.arange(n_pits + 1)
    final = np.full(n_pits + 1, UNSET, np.int64)
    spill = np.full(n_pits + 1, np.nan)
    outlet = np.full(n_pits + 1, -1, np.int64)
    # order: lowest spill, then steepest, then (exact ties) inside cell in raster order, then direction
    order = np.argsort(e_k[:n], kind="mergesort")
    order = order[np.argsort(e_cell[:n][order], kind="mergesort")]
    order = order[np.argsort(-e_slope[:n][order], kind="mergesort")]
    order = order[np.argsort(e_z[:n][order], kind="mergesort")]
    for t in order:
        a = find(parent, e_pit[t])
        if final[a] != UNSET:
            continue                                        # already drained
        to = e_to[t]
        if to < 0:                                          # crossing into another pit
            b = find(parent, -to)
            if a == b:
                continue
            if final[b] == UNSET:                           # both still closed: they merge
                for q in (a, b):
                    if np.isnan(spill[q]):
                        spill[q], outlet[q] = e_z[t], e_cell[t]
                parent[a] = b
                continue
            to = final[b]
        if np.isnan(spill[a]):
            spill[a], outlet[a] = e_z[t], e_cell[t]
        final[a] = to

    drains_to = np.empty(n_pits + 1, np.int64)
    for p in range(1, n_pits + 1):
        drains_to[p] = final[find(parent, p)]
    l = lab.ravel()
    for i in range(nr * nc):
        v = l[i]
        if v < 0 and v != NODATA and drains_to[-v] != UNSET:
            l[i] = drains_to[-v]
    return spill[1:], outlet[1:], drains_to[1:]

## Pass 1

In [ ]:
out = Path(OUT_DIR)
out.mkdir(parents=True, exist_ok=True)

with rasterio.open(DEM_PATH) as src:
    z = src.read(1).astype(np.float64 if src.dtypes[0] == "float64" else np.float32, copy=False)
    if src.nodata is not None and not np.isnan(src.nodata):
        z[z == src.nodata] = np.nan
    transform, crs, (dx, dy) = src.transform, src.crs, src.res
    profile = dict(driver="GTiff", height=src.height, width=src.width, count=1, dtype="int32", crs=crs,
                   transform=transform, nodata=NODATA, tiled=True, compress="deflate", BIGTIFF="IF_SAFER")
cell_area = dx * dy

wl = gpd.read_file(WETLANDS_PATH, layer=WETLANDS_LAYER)
wl = wl[wl.geometry.notna() & ~wl.geometry.is_empty & wl[ID_FIELD].notna()].to_crs(crs)
wid = wl[ID_FIELD].to_numpy()
if wid.min() < 0 or np.any(wid != np.floor(wid)):
    raise ValueError(f"'{ID_FIELD}' must hold non-negative integers")
wl = wl.loc[wl.area.sort_values(ascending=False).index]      # smaller polygons burn last, so they win overlaps

lab = np.full(z.shape, UNSET, dtype=np.int32)
features.rasterize(zip(wl.geometry, wl[ID_FIELD].astype(np.int64) + 1), out=lab, transform=transform,
                   all_touched=ALL_TOUCHED)
if STREAMS_PATH:
    streams = gpd.read_file(STREAMS_PATH).to_crs(crs)
    features.rasterize(((g, EXTERNAL) for g in streams.geometry if g is not None), out=lab,
                       transform=transform, all_touched=True)
lab[np.isnan(z)] = NODATA
n_bins = int(wid.max()) + 2
wetland_cells = np.bincount(lab[lab > 0], minlength=n_bins)

n_pits, pit_cell, pit_floor = pass1(z, lab, dx, dy)

with rasterio.open(out / f"{PREFIX}_pass1.tif", "w", **profile) as dst:
    dst.write(lab, 1)

pass1_cells = np.zeros(n_bins, np.int64)
pit_cells = np.zeros(n_pits + 1, np.int64)
for r0 in range(0, lab.shape[0], 1024):
    a = lab[r0:r0 + 1024]
    pass1_cells += np.bincount(a[a >= 0], minlength=n_bins)
    pit_cells += np.bincount(-a[(a < 0) & (a != NODATA)], minlength=n_pits + 1)
n_valid = pass1_cells.sum() + pit_cells.sum()
print(f"{n_valid:,} valid cells: {pass1_cells[1:].sum() / n_valid:.1%} reach a wetland, "
      f"{pass1_cells[0] / n_valid:.1%} leave the DEM, {pit_cells.sum() / n_valid:.1%} end in one of "
      f"{n_pits:,} pits")

## Pass 2: drain the pits

In [ ]:
spill, outlet, drains_to = pass2(z, lab, n_pits, dx, dy)

with rasterio.open(out / f"{PREFIX}_labels.tif", "w", **profile) as dst:
    dst.write(lab, 1)

final_cells = np.zeros(n_bins, np.int64)
for r0 in range(0, lab.shape[0], 1024):
    a = lab[r0:r0 + 1024]
    final_cells += np.bincount(a[a >= 0], minlength=n_bins)
print(f"After draining pits: {final_cells[1:].sum() / n_valid:.1%} reach a wetland, "
      f"{final_cells[0] / n_valid:.1%} leave the DEM, {(lab < 0).sum() - (lab == NODATA).sum():,} cells unresolved")

pit_r, pit_c = np.divmod(pit_cell, lab.shape[1])
out_r, out_c = np.divmod(outlet, lab.shape[1])
pits = pd.DataFrame({
    "pit_id": np.arange(1, n_pits + 1), "floor_row": pit_r, "floor_col": pit_c,
    "floor_elevation": z[pit_r, pit_c], "floor_cells": pit_floor, "catchment_cells": pit_cells[1:],
    "outlet_row": out_r, "outlet_col": out_c, "spill_elevation": spill,
    "drains_to": drains_to,            # label value: wetland ID + 1, or 0
})
pits.to_csv(out / f"{PREFIX}_pits.csv", index=False)

ids = np.unique(wid).astype(np.int64)
summary = pd.DataFrame({
    ID_FIELD: ids, "label_value": ids + 1, "wetland_cells": wetland_cells[ids + 1],
    "drainage_cells_pass1": pass1_cells[ids + 1], "drainage_cells": final_cells[ids + 1],
})
summary["drainage_area_m2"] = summary["drainage_cells"] * cell_area
summary.to_csv(out / f"{PREFIX}_summary.csv", index=False)
summary.sort_values("drainage_cells", ascending=False).head(10)

## Optional: drainage-area polygons

In [ ]:
if WRITE_POLYGONS:
    pieces = []
    for r0 in range(0, lab.shape[0], 4096):
        a = lab[r0:r0 + 4096]
        pieces += [(int(v), shape(g)) for g, v in features.shapes(
            a, mask=a > 0, transform=transform * rasterio.Affine.translation(0, r0))]
    polys = gpd.GeoDataFrame({"label_value": [v for v, _ in pieces]},
                             geometry=[g for _, g in pieces], crs=crs).dissolve("label_value").reset_index()
    polys = polys.merge(summary, on="label_value")
    polys.to_file(out / f"{PREFIX}_drainage_areas.gpkg", driver="GPKG")